# cGAN으로 생성 제어하기

In [1]:
import os # 운영체제 상호작용을 위한 라이브러리
import torch # 파이토치 핵심 라이브러리
import torchvision # 컴퓨터 비전 데이터 및 도구
import torch.nn as nn # 신경망 구축을 위한 모듈
import torch.optim as optim # 최적화 알고리즘
from torchvision import transforms, datasets # 데이터셋 및 변환 도구
from torchvision.utils import save_image # 이미지 저장 유틸리티
import matplotlib.pyplot as plt # 결과 시각화 도구
import numpy as np # 수치 계산 및 배열 조작

In [2]:
# 하이퍼파라미터 # 학습 관련 설정값
EPOCHS = 300 # 전체 데이터셋 반복 학습 횟수
BATCH_SIZE = 100 # 한 번에 처리할 데이터 묶음 크기
USE_CUDA = torch.cuda.is_available() # GPU 사용 가능 여부 확인
DEVICE = torch.device("cuda" if USE_CUDA else "cpu") # 사용할 연산 장치 설정
print("Using Device:", DEVICE) # 현재 사용 장치 출력

In [3]:
# Fashion MNIST 데이터셋 # 데이터 준비 섹션
trainset = datasets.FashionMNIST( # 패션 아이템 이미지 데이터셋 로드
    './.data', # 데이터 저장 경로
    train=True, # 학습용 데이터 사용
    download=True, # 없으면 자동 다운로드
    transform=transforms.Compose([ # 이미지 전처리 과정 정의
       transforms.ToTensor(), # 텐서 형태로 변환
       transforms.Normalize((0.5,), (0.5,)) # -1에서 1 사이로 정규화
    ])
)
train_loader = torch.utils.data.DataLoader( # 데이터를 배치 단위로 공급하는 로더
    dataset     = trainset, # 대상 데이터셋
    batch_size  = BATCH_SIZE, # 배치 크기 설정
    shuffle     = True # 데이터를 무작위로 섞음
)

In [4]:
# 생성자 (Generator) # 가짜 이미지를 만드는 신경망 정의
class Generator(nn.Module): # nn.Module 상속
    def __init__(self): # 초기화 함수
        super().__init__() # 부모 클래스 초기화
        
        self.embed = nn.Embedding(10, 10) # 10개의 레이블을 각각 10차원 벡터로 변환
        
        self.model = nn.Sequential( # 신경망 층을 순서대로 쌓음
            nn.Linear(110, 256), # 노이즈(100) + 레이블(10) 입력 -> 256 노드
            nn.LeakyReLU(0.2, inplace=True), # 활성화 함수 LeakyReLU
            nn.Linear(256, 512), # 256 -> 512 노드
            nn.LeakyReLU(0.2, inplace=True), # 활성화 함수
            nn.Linear(512, 1024), # 512 -> 1024 노드
            nn.LeakyReLU(0.2, inplace=True), # 활성화 함수
            nn.Linear(1024, 784), # 1024 -> 784(이미지 크기) 노드
            nn.Tanh() # 출력값을 -1 ~ 1 사이로 압축
        )
    
    def forward(self, z, labels): # 순전파 함수
        c = self.embed(labels) # 레이블 임베딩 수행
        x = torch.cat([z, c], 1) # 노이즈와 레이블 정보를 하나로 합침
        return self.model(x) # 모델을 통과시켜 이미지 생성

In [ ]:
# 판별자 (Discriminator) # 진짜/가짜를 구별하는 신경망 정의
class Discriminator(nn.Module): # nn.Module 상속
    def __init__(self): # 초기화 함수
        super().__init__() # 부모 클래스 초기화
        
        self.embed = nn.Embedding(10, 10) # 레이블 정보를 위한 임베딩
        
        self.model = nn.Sequential( # 신경망 층 구성
            nn.Linear(794, 1024), # 이미지(784) + 레이블(10) 입력 -> 1024 노드
            nn.LeakyReLU(0.2, inplace=True), # 활성화 함수
            nn.Dropout(0.3), # 과적합 방지를 위해 30% 노드 비활성화
            nn.Linear(1024, 512), # 1024 -> 512 노드
            nn.LeakyReLU(0.2, inplace=True), # 활성화 함수
            nn.Dropout(0.3), # 드롭아웃
            nn.Linear(512, 256), # 512 -> 256 노드
            nn.LeakyReLU(0.2, inplace=True), # 활성화 함수
            nn.Dropout(0.3), # 드롭아웃
            nn.Linear(256, 1), # 256 -> 1 (최종 판별 결과)
            nn.Sigmoid() # 0(가짜)에서 1(진짜) 사이의 확률로 출력
        )
    
    def forward(self, x, labels): # 순전파 함수
        c = self.embed(labels) # 레이블 임베딩 수행
        x = torch.cat([x, c], 1) # 이미지 데이터와 레이블 정보를 합침
        return self.model(x) # 모델을 통과시켜 판별 결과 반환

In [6]:
# 모델 인스턴스를 만들고 모델의 가중치를 지정한 장치로 보내기 # 모델 준비 섹션
D = Discriminator().to(DEVICE) # 판별자 생성 및 장치 할당
G = Generator().to(DEVICE) # 생성자 생성 및 장치 할당

# 이진 교차 엔트로피 함수와 # 손실 함수 및 최적화기 설정
# 생성자와 판별자를 최적화할 Adam 모듈
criterion = nn.BCELoss() # 바이너리 크로스 엔트로피 손실 함수
d_optimizer = optim.Adam(D.parameters(), lr =0.0002) # 판별자용 Adam 최적화기
g_optimizer = optim.Adam(G.parameters(), lr =0.0002) # 생성자용 Adam 최적화기

In [7]:
total_step = len(train_loader) # 전체 배치 횟수
for epoch in range(EPOCHS): # 설정된 에폭만큼 반복 학습
    for i, (images, labels) in enumerate(train_loader): # 배치 단위로 데이터 순회
        images = images.reshape(BATCH_SIZE, -1).to(DEVICE) # 이미지를 벡터로 평탄화하고 장치로 전송
        
        # '진짜'와 '가짜' 레이블 생성 # 학습 정답지 준비
        real_labels = torch.ones(BATCH_SIZE, 1).to(DEVICE) # 진짜 데이터 정답은 1
        fake_labels = torch.zeros(BATCH_SIZE, 1).to(DEVICE) # 가짜 데이터 정답은 0

        # 판별자가 진짜 이미지를 진짜로 인식하는 오차 계산 (데이터셋 레이블 입력) # 판별자 학습 (진짜 데이터)
        labels = labels.to(DEVICE) # 레이블 정보 전송
        outputs = D(images, labels) # 판별자의 예측값 계산
        d_loss_real = criterion(outputs, real_labels) # 진짜 데이터에 대한 손실 계산
        real_score = outputs # 나중에 평균 출력을 위해 저장
    
        # 무작위 텐서와 무작위 레이블을 생성자에 입력해 가짜 이미지 생성 # 가짜 이미지 생성
        z = torch.randn(BATCH_SIZE, 100).to(DEVICE) # 노이즈 생성
        g_label = torch.randint(0, 10, (BATCH_SIZE,)).to(DEVICE) # 가짜용 무작위 레이블 생성
        fake_images = G(z, g_label) # 생성자를 통해 가짜 이미지 생성
        
        # 판별자가 가짜 이미지를 가짜로 인식하는 오차 계산 # 판별자 학습 (가짜 데이터)
        outputs = D(fake_images, g_label) # 가짜 이미지에 대한 판별 예측
        d_loss_fake = criterion(outputs, fake_labels) # 가짜 데이터에 대한 손실 계산
        fake_score = outputs # 점수 저장
        
        # 진짜와 가짜 이미지를 갖고 낸 오차를 더해서 판별자의 오차 계산 # 판별자 최종 손실
        d_loss = d_loss_real + d_loss_fake
        
        # 역전파 알고리즘으로 판별자 모델의 학습을 진행 # 판별자 가중치 업데이트
        d_optimizer.zero_grad() # 기울기 초기화
        g_optimizer.zero_grad() # 생성자 기울기 초기화 (간섭 방지)
        d_loss.backward() # 역전파 수행
        d_optimizer.step() # 판별자 업데이트
        
        # 생성자가 판별자를 속였는지에 대한 오차 계산(무작위 레이블 입력) # 생성자 학습
        fake_images = G(z, g_label) # 가짜 이미지 재생성
        outputs = D(fake_images, g_label) # 판별자에 다시 입력
        g_loss = criterion(outputs, real_labels) # 가짜를 진짜(1)로 속였는지 손실 계산

        # 역전파 알고리즘으로 생성자 모델의 학습을 진행 # 생성자 가중치 업데이트
        d_optimizer.zero_grad() # 기울기 초기화
        g_optimizer.zero_grad() # 기울기 초기화
        g_loss.backward() # 역전파 수행
        g_optimizer.step() # 생성자 업데이트
        
    print('이폭 [{}/{}] d_loss:{:.4f} g_loss: {:.4f} D(x):{:.2f} D(G(z)):{:.2f}' # 주기적인 결과 출력
          .format(epoch,
                  EPOCHS,
                  d_loss.item(),
                  g_loss.item(),
                  real_score.mean().item(),
                  fake_score.mean().item()))

In [8]:
# 만들고 싶은 아이템 생성하고 시각화하기 # 결과물 확인 섹션
item_number = 9 # 생성하고자 하는 아이템 번호 설정
z = torch.randn(1, 100).to(DEVICE) # 한 개의 무작위 노이즈 벡터 생성
g_label = torch.full((1,), item_number, dtype=torch.long).to(DEVICE) # 지정된 번호의 레이블 텐서 생성
sample_images = G(z, g_label) # 모델을 사용해 이미지 생성

sample_images_img = np.reshape(sample_images.data.cpu().numpy() # 텐서를 넘파이 배열로 변환
                               [0],(28, 28)) # 28x28 이미지 형태로 재구성
plt.imshow(sample_images_img, cmap = 'gray') # 회색조 이미지로 표시 설정
plt.show() # 이미지 화면 출력